# Local continuous-data gap analysis — what must be downloaded before detection

For every station in the Gyeongju region, find the days a station was **operating** (metadata) but the continuous
waveforms are **not on local disk**, then write concrete **download worklists**. When these gaps are filled, the
deep-learning detection pipeline starts.

**Metadata sources (all now epoch-resolved):**

| network | source | epochs |
|---|---|---|
| **KS + KG** | **`KS_KG_metadata_1.0.2.xml`** (FDSN StationXML, 547 codes) | real **channel-level** epochs, incl. *retired* codes (e.g. USN→USN2, DAG→DAG2) |
| **NS** (GHBSN dense) | `GHBSN_info_ver202312_modified.csv` | start/end epochs (open = today) |

This replaces the old flat `station_update.dat` + per-year used-lists for KS — those had no epochs and only current
codes, so retired stations (and their local data) were invisible. **Operating window** per station = span of its
**seismometer** channels (velocity/short-period); co-located accelerometers (`?G`/`?N`) are left open to 2099 in
the metadata, so they are excluded from the window (their real deployment end is not recorded). Accelerometer-only
stations are kept and flagged (`accel_only`) — usable but secondary for velocity-based DL picking.

**Gap** (per station, per day): `missing = (days inside the operating epoch, up to today) − (days with a local
file)`. A day counts as present if ≥ 1 file exists (component completeness is a later, finer check).
**Outputs:** `gaps_confirmed_<NET>.csv` + `download_worklist_KS_KG.csv` + visualizations.

In [ ]:
# ---------------------------------------------------------------- parameters (edit here)
import os, glob
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.dates as mdates, matplotlib.font_manager as fm
import xml.etree.ElementTree as ET
from collections import defaultdict

# --- REGION DEFINITION (see Sec 1b for the consistency check vs the stage-1 KS/KG work) ---
#   Gyeongju city ...... (35.856, 129.224)   <- current default (administrative reference)
#   2016 M5.8 mainshock  (35.770, 129.190)   <- seismological reference for the Gyeongju sequence
#   stage-1 UF-box centre(35.750, 129.400)   <- what the KS/KG relocation used (UF_BOX centre)
REGION_CENTER=(35.856,129.224)        # <-- edit here to change how "Gyeongju region" is defined
RMAX_KM=100.0
T0=pd.Timestamp("2010-01-01"); TODAY=pd.Timestamp.today().normalize()
ACCEL=("G","N")                       # 2nd channel-code char = accelerometer -> excluded from the operating window

ROOT="/home/msseo/works/02.Ulsan_Fault_detection"
NECIS="/home/msseo/works/Claude/data/necis/continuous"
OUT=os.path.join(ROOT,"Gyeongju_catalog")
def resolve(*cands):
    for c in cands:
        if os.path.exists(c): return c
    raise FileNotFoundError(cands)
P_XML =resolve(f"{ROOT}/KS_KG/local_magnitudes/responses/master/KS_KG_metadata_1.0.2.xml",
               f"{ROOT}/stage1_KS_KG_exploration/KS_KG/local_magnitudes/responses/master/KS_KG_metadata_1.0.2.xml")
P_NSCSV=resolve(f"{ROOT}/GHBSN_metadata/20240715/GHBSN_info_ver202312_modified.csv")
P_ST1  =resolve(f"{ROOT}/KS_KG/continuous",f"{ROOT}/stage1_KS_KG_exploration/KS_KG/continuous")

_av={f.name for f in fm.fontManager.ttflist}
for _f in ("Helvetica","Arial","Nimbus Sans","TeX Gyre Heros","DejaVu Sans"):
    if _f in _av: mpl.rcParams["font.family"]=_f; break
mpl.rcParams.update({"figure.dpi":130,"axes.grid":True,"grid.alpha":0.3,"font.size":10,
                     "legend.framealpha":1,"legend.edgecolor":"black","legend.facecolor":"white","axes.unicode_minus":False})
def hav_km(lat1,lon1,lat2,lon2):
    a=(np.sin(np.radians(lat2-lat1)/2)**2+np.cos(np.radians(lat1))*np.cos(np.radians(lat2))
       *np.sin(np.radians(lon2-lon1)/2)**2)
    return 2*6371.0*np.arcsin(np.sqrt(a))
GYEONGJU=REGION_CENTER
print(f"region: 100 km around REGION_CENTER {REGION_CENTER} | window {T0.date()} .. {TODAY.date()}")

## 1 · Station metadata with real epochs (KS+KG from StationXML, NS from GHBSN)

In [ ]:
NSXML="{http://www.fdsn.org/xml/station/1}"
def _ts(x):
    try: return pd.Timestamp(x).tz_localize(None)
    except Exception: return None                                   # 9999-12-31 etc -> open
def load_stationxml(path,accel=ACCEL):
    """Per (net,code): coords + operating window from SEISMOMETER channel epochs (accel ?G/?N excluded)."""
    A=defaultdict(lambda:{"lat":None,"lon":None,"seis":[],"all":[],"bands":set()}); net=None
    for ev,el in ET.iterparse(path,events=("start","end")):
        t=el.tag.split("}")[-1]
        if ev=="start" and t=="Network": net=el.get("code")
        elif ev=="end" and t=="Station":
            d=A[(net,el.get("code"))]; la=el.findtext(NSXML+"Latitude")
            if la: d["lat"]=float(la); d["lon"]=float(el.findtext(NSXML+"Longitude"))
            for c in el.findall(NSXML+"Channel"):
                cc=c.get("code") or ""; d["bands"].add(cc[:2]); iv=(c.get("startDate"),c.get("endDate"))
                d["all"].append(iv)
                if len(cc)>1 and cc[1] not in accel: d["seis"].append(iv)
            el.clear()
    rows=[]
    for (net,code),d in A.items():
        if d["lat"] is None: continue
        use=d["seis"] or d["all"]
        st=[x for x in (_ts(s) for s,e in use if s) if x is not None]
        en=[x for x in (_ts(e) for s,e in use if e) if x is not None]
        maxen=max(en) if en else None
        rows.append(dict(net=net,sta=code,lat=d["lat"],lon=d["lon"],
                         t0=min(st) if st else T0,
                         t1=(TODAY if (maxen is None or maxen.year>=2098) else maxen),
                         accel_only=(len(d["seis"])==0),has_hh=("HH" in d["bands"]),
                         bands="/".join(sorted(d["bands"]))))
    return pd.DataFrame(rows)
_xml=load_stationxml(P_XML); _xml["dist_km"]=hav_km(REGION_CENTER[0],REGION_CENTER[1],_xml.lat,_xml.lon)
_xml=_xml[_xml.dist_km<=RMAX_KM]
kg=_xml[_xml.net=="KG"].copy(); ks=_xml[_xml.net=="KS"].copy()
# NS (GHBSN dense) from its info CSV; U*/Y* = Uljin/Jeonbuk deployments -> out of region
nsm=pd.read_csv(P_NSCSV).rename(columns={"station":"sta","stla":"lat","stlo":"lon"})
nsm["t0"]=pd.to_datetime(nsm.starttime); nsm["t1"]=pd.to_datetime(nsm.endtime)
nsm.loc[nsm.t1.dt.year>=2099,"t1"]=TODAY
nsm["dist_km"]=hav_km(REGION_CENTER[0],REGION_CENTER[1],nsm.lat,nsm.lon)
nsm=nsm[(nsm.dist_km<=RMAX_KM)&(~nsm.sta.str.startswith(("U","Y")))].drop_duplicates("sta")
nsm["accel_only"]=False; nsm["has_hh"]=True
for d in (kg,ks): d["net"]=d["net"]
print(f"in-region stations (100 km): KG {len(kg)} | KS {len(ks)} | NS {len(nsm)}")
print(f"  KS velocity/seismometer {int((~ks.accel_only).sum())} | accelerometer-only {int(ks.accel_only.sum())} "
      f"(kept, flagged; secondary for DL velocity picking)")
print(f"  retired-code example: KS.USN {ks.loc[ks.sta=='USN','t1'].iloc[0].date() if (ks.sta=='USN').any() else '-'} "
      f"-> KS.USN2 from {ks.loc[ks.sta=='USN2','t0'].iloc[0].date() if (ks.sta=='USN2').any() else '-'}")

## 1b · How "Gyeongju region" is defined — consistency with the stage-1 KS/KG work

Both stages use a **100 km radius**; only the centre differs. Stage-1 KS/KG relocation centred on the **Ulsan-Fault
box centre (35.75, 129.40)** (`UF_BOX` centre in `make_uf_catalog.py`); this notebook defaults to **Gyeongju city**.
The cell quantifies the offset and lists stations in one disk but not the other.

In [ ]:
UF_CENTER=(35.75,129.40); MS2016=(35.770,129.190)
print(f"centre offsets from REGION_CENTER {REGION_CENTER}: stage-1 UF-box {hav_km(*REGION_CENTER,*UF_CENTER):.1f} km | 2016 mainshock {hav_km(*REGION_CENTER,*MS2016):.1f} km")
_all=pd.concat([kg[["net","sta","lat","lon"]],ks[["net","sta","lat","lon"]],
                nsm.assign(net="NS")[["net","sta","lat","lon"]]],ignore_index=True).dropna(subset=["lat","lon"])
_all["d_new"]=hav_km(REGION_CENTER[0],REGION_CENTER[1],_all.lat,_all.lon)
_all["d_uf"]=hav_km(UF_CENTER[0],UF_CENTER[1],_all.lat,_all.lon)
_in=_all.d_new<=RMAX_KM; _iu=_all.d_uf<=RMAX_KM
print(f"stations within 100 km: current-centre {int(_in.sum())} | stage-1 UF-centre {int(_iu.sum())} | both {int((_in&_iu).sum())}")
print(f"  only in current (Gyeongju) disk: {' '.join((_all[_in&~_iu].net+'.'+_all[_in&~_iu].sta))}")
print(f"  only in stage-1 (UF) disk       : {' '.join((_all[_iu&~_in].net+'.'+_all[_iu&~_in].sta)) or '(none)'}")
print("  -> set REGION_CENTER=UF_CENTER in the parameters cell to reproduce the stage-1 footprint exactly.")

## 1c · Metadata station-availability matrix (station × year)

Whether each station was operating that year per metadata: **1** = operating (epoch overlaps the year), **0** = not.
With real epochs from the StationXML there is no longer an "unknown" category for KS. Saved to
`metadata_station_year_matrix.csv`.

In [ ]:
YEARS=list(range(2010,TODAY.year+1))
def _yr(t0,t1,Y):
    a=max(t0,pd.Timestamp(f"{Y}-01-01")); b=min(t1,pd.Timestamp(f"{Y}-12-31"),TODAY); return 1 if b>=a else 0
def _mrow(net,r): 
    d={"net":net,"sta":r.sta,"dist_km":round(r.dist_km,1),"accel_only":bool(r.accel_only)}
    d.update({Y:_yr(r.t0,r.t1,Y) for Y in YEARS}); return d
rows=[_mrow("KG",r) for _,r in kg.iterrows()]+[_mrow("KS",r) for _,r in ks.iterrows()]+[_mrow("NS",r) for _,r in nsm.iterrows()]
AVAIL=pd.DataFrame(rows).sort_values(["net","dist_km"]).reset_index(drop=True)
AVAIL.to_csv(os.path.join(OUT,"metadata_station_year_matrix.csv"),index=False)
print("operating stations per year (metadata):")
print(AVAIL.groupby("net")[YEARS].sum().astype(int).to_string())
print(f"\nwrote metadata_station_year_matrix.csv  ({len(AVAIL)} stations x {len(YEARS)} years)")
AVAIL.head(10)

### 1c-year · Stations operating each year, listed per network (full lists)

In [ ]:
NETS=["KG","KS","NS"]
def _codes(net,Y): 
    sub=AVAIL[AVAIL.net==net]; return list(sub.loc[sub[Y]==1,"sta"])   # AVAIL sorted nearest-first
BY=pd.DataFrame(index=YEARS); BY.index.name="year"
for net in NETS:
    BY[f"{net}_n"]=[len(_codes(net,Y)) for Y in YEARS]
    BY[net]=[(", ".join(_codes(net,Y)) or "-") for Y in YEARS]
BY.to_csv(os.path.join(OUT,"metadata_stations_by_year.csv"))
print("wrote metadata_stations_by_year.csv\n")
for Y in YEARS:
    print(f"{Y}:")
    for net in NETS: print(f"    {net}({BY.loc[Y,net+'_n']}): {', '.join(_codes(net,Y)) or '-'}")
    print()
pd.set_option("display.max_colwidth",None); BY[["KG_n","KG","KS_n","KS","NS_n"]]

### 1c-plot · availability heatmap

In [ ]:
import matplotlib.colors as mcol
M=AVAIL.sort_values(["net","dist_km"]).reset_index(drop=True); Z=M[YEARS].astype(float).values
fig,ax=plt.subplots(figsize=(9,13))
im=ax.imshow(Z,aspect="auto",cmap=mcol.ListedColormap(["#f0f0f0","#2b7bba"]),vmin=0,vmax=1,interpolation="nearest")
ax.set_xticks(range(len(YEARS)),YEARS,rotation=90,fontsize=8)
ax.set_yticks(range(len(M)),(M.net+"."+M.sta),fontsize=4.5)
for b in np.where(M.net.values[1:]!=M.net.values[:-1])[0]: ax.axhline(b+0.5,color="k",lw=0.8)
ax.set(title="Metadata station availability by year (blue=operating, grey=not) — KS/KG epochs from StationXML")
fig.tight_layout(); plt.show()

## 2 · Local-data day sets (archive scans, merged)

Filename scans over `NS/`, the stage-1 `KS_KG/continuous/`, and the NECIS archive; merged into `LOCAL[(net,sta)]`.
With real epochs the retired codes (USN, DAG) now match a metadata row, so their local data are no longer orphaned.

In [ ]:
def scan_sds(root,prefixes=("HH",)):
    out={}
    if not os.path.isdir(root): return out
    for sta in sorted(os.listdir(root)):
        sd=os.path.join(root,sta)
        if not os.path.isdir(sd): continue
        days=set()
        for ch in os.listdir(sd):
            if not ch.endswith(".D") or not ch.startswith(tuple(prefixes)): continue
            for f in os.scandir(os.path.join(sd,ch)):
                p=f.name.split(".")
                if len(p)>=7 and p[-2].isdigit() and p[-1].isdigit(): days.add((int(p[-2]),int(p[-1])))
        if days: out[sta]=days
    return out
def scan_necis(root):
    out={}
    if not os.path.isdir(root): return out
    for ydir in sorted(glob.glob(os.path.join(root,"20??"))):
        for sd in os.scandir(ydir):
            if not sd.is_dir(): continue
            days=out.setdefault(sd.name,set())
            for f in os.scandir(sd.path):
                p=f.name.split(".")
                if len(p)>=5 and p[3].isdigit() and p[4].isdigit(): days.add((int(p[3]),int(p[4])))
    return out
def to_dates(days): return {pd.Timestamp(f"{a}-01-01")+pd.Timedelta(days=b-1) for a,b in days}
DATA_NS={s:to_dates(d) for s,d in scan_sds(os.path.join(ROOT,"NS")).items()}
D1=scan_sds(P_ST1,prefixes=("HH","HG","EL","SH")); D2=scan_necis(NECIS)
KGSET=set(kg.sta); DATA_KS={}; DATA_KG={}
for src in (D1,D2):
    for s,d in src.items(): (DATA_KG if s in KGSET else DATA_KS).setdefault(s,set()).update(to_dates(d))
LOCAL={}
for net,store in [("NS",DATA_NS),("KS",DATA_KS),("KG",DATA_KG)]:
    for s,d in store.items(): LOCAL[(net,s)]=d
print(f"local day-sets: NS {len(DATA_NS)} | KS {len(DATA_KS)} | KG {len(DATA_KG)} stations")
print(f"  retired codes now matched to metadata: "+", ".join(f"{s}({min(LOCAL[('KS',s)]).year}-{max(LOCAL[('KS',s)]).year})" for s in ('USN','DAG') if ('KS',s) in LOCAL))

## 3 · Compute the gaps (all networks now epoch-resolved -> all confirmed)

In [ ]:
def spans(dates):
    if not dates: return []
    ds=sorted(dates); out=[[ds[0],ds[0]]]
    for d in ds[1:]:
        if (d-out[-1][1]).days<=1: out[-1][1]=d
        else: out.append([d,d])
    return [(a,b) for a,b in out]
def daterange(a,b): return set(pd.date_range(max(a,T0),min(b,TODAY),freq="D"))
GAPS=[]
for net,meta,store in [("KG",kg,DATA_KG),("KS",ks,DATA_KS),("NS",nsm,DATA_NS)]:
    for _,r in meta.iterrows():
        miss=daterange(r.t0,r.t1)-store.get(r.sta,set())
        for a,b in spans(miss): GAPS.append((net,r.sta,a,b,(b-a).days+1))
G=pd.DataFrame(GAPS,columns=["net","sta","start","end","n_days"])
print(G.groupby("net").n_days.agg(missing_days="sum",n_periods="count").to_string())
print(f"\nCONFIRMED missing station-days total: {int(G.n_days.sum())}")

### 3b · Non-local stations per year — the to-download list

Among the operating stations of §1c-year, the ones with **no local waveforms for that year** (operated per
metadata, but zero files on disk in that year — the clean "download this station for this year" targets;
partial-year gaps are in the day-resolution `gaps_confirmed_*` CSVs). Sorted nearest-to-Gyeongju first. Saved to
`missing_stations_by_year.csv`.

In [ ]:
LOCAL_YEARS={k:{d.year for d in v} for k,v in LOCAL.items()}
def _miss(net,Y):
    sub=AVAIL[AVAIL.net==net]
    return [s for s in sub.loc[sub[Y]==1,"sta"] if Y not in LOCAL_YEARS.get((net,s),set())]   # operating but not on disk that year
BYMISS=pd.DataFrame(index=YEARS); BYMISS.index.name="year"
for net in NETS:
    BYMISS[f"{net}_n"]=[len(_miss(net,Y)) for Y in YEARS]
    BYMISS[net]=[(", ".join(_miss(net,Y)) or "-") for Y in YEARS]
BYMISS.to_csv(os.path.join(OUT,"missing_stations_by_year.csv"))
print("wrote missing_stations_by_year.csv  (operating-but-not-local, per year)\n")
for Y in YEARS:
    print(f"{Y}  (missing KG {BYMISS.loc[Y,'KG_n']} / KS {BYMISS.loc[Y,'KS_n']} / NS {BYMISS.loc[Y,'NS_n']}):")
    for net in NETS: print(f"    {net}: {', '.join(_miss(net,Y)) or '-'}")
    print()
BYMISS[["KG_n","KG","KS_n","KS","NS_n"]]

## 4 · Gap visualizations

In [ ]:
yrs=np.arange(2010,TODAY.year+1)
fig,ax=plt.subplots(figsize=(11,4.2)); w=0.27
for i,(net,colr) in enumerate([("KS","#1f77b4"),("KG","#ff7f0e"),("NS","#2ca02c")]):
    per=np.zeros(len(yrs))
    for _,r in G[G.net==net].iterrows():
        for y in range(r.start.year,r.end.year+1):
            a=max(r.start,pd.Timestamp(f"{y}-01-01")); b=min(r.end,pd.Timestamp(f"{y}-12-31")); per[y-2010]+=(b-a).days+1
    ax.bar(yrs+(i-1)*w,per,w,color=colr,label=net)
ax.set(xlabel="year",ylabel="missing station-days",title="Confirmed local-data gaps by year and network"); ax.legend()
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(12.5,11)); y=0; ylbl=[]
def bar(a,b,yy,c,h=0.75,z=2): ax.barh(yy,(b-a).days+1,left=a,height=h,color=c,zorder=z,lw=0)
for net,meta,store in [("KG",kg,DATA_KG),("KS",ks,DATA_KS)]:
    for _,r in meta.sort_values("dist_km",ascending=False).iterrows():
        for a,b in spans(store.get(r.sta,set())): bar(a,b,y,"#69b578",z=3)
        for _,q in G[(G.net==net)&(G.sta==r.sta)].iterrows(): bar(q.start,q.end,y,"#d62728",h=0.5)
        ylbl.append(f"{net} {r.sta}"); y+=1
y0=y
for _,r in nsm.sort_values("t0").iterrows():
    for a,b in spans(DATA_NS.get(r.sta,set())): bar(a,b,y,"#69b578",h=0.95,z=3)
    for _,q in G[(G.net=="NS")&(G.sta==r.sta)].iterrows(): bar(q.start,q.end,y,"#d62728",h=0.6)
    y+=1
ax.text(T0,(y0+y)/2,f"  NS x {len(nsm)}",va="center",fontsize=9,color="#1b5e20")
ax.set_yticks(range(len(ylbl)),ylbl,fontsize=6); ax.set_ylim(-1,y); ax.set_xlim(T0,TODAY)
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set(title="Local-data gap map — green: on disk, red: confirmed missing"); plt.tight_layout(); plt.show()

## 5 · Download worklists and summary

In [ ]:
for net in NETS:
    gg=G[G.net==net].sort_values(["sta","start"]); gg.to_csv(os.path.join(OUT,f"gaps_confirmed_{net}.csv"),index=False)
    print(f"wrote gaps_confirmed_{net}.csv  ({len(gg)} periods, {int(gg.n_days.sum())} station-days)")
wl=G[G.net.isin(["KS","KG"])].sort_values(["net","sta","start"]).copy()
wl["source"]=np.where(wl.net=="KS","NECIS","NECIS-or-kquake")
wl.to_csv(os.path.join(OUT,"download_worklist_KS_KG.csv"),index=False)
print(f"wrote download_worklist_KS_KG.csv  ({len(wl)} request periods)")
print("\nNEXT: fill the gaps (NECIS batch: ~/works/Claude/archive_necis.sh; NS 2024-> from the GHBSN holder),")
print("re-run until confirmed-missing ~ 0, then start the DL detection pipeline (03.*).")